In [ ]:
!pip install Biopython

In [ ]:
!pip install sentence_transformers

In [ ]:
import re
import time
import numpy as np
from Bio import Entrez
from sentence_transformers import SentenceTransformer

Entrez.email = "example@com"
ranker = SentenceTransformer("pritamdeka/S-PubMedBert-MS-MARCO")

STOPWORDS = {
    "the", "and", "for", "with", "from", "that", "this", "these", "those", "are", "was", "were",
    "been", "being", "have", "has", "had", "can", "could", "should", "would", "may", "might",
    "must", "will", "not", "but", "also", "such", "than", "then", "there", "their", "which",
    "who", "what", "when", "where", "while", "into", "over", "under", "about", "after", "before",
    "between", "during", "within", "without", "because", "however", "therefore", "other",
    "more", "most", "some", "any", "all", "each", "only", "very", "both", "either", "its",
    "patient", "patients", "clinical", "treatment", "therapy", "disease", "condition",
    "recommended", "recommend", "consider", "important", "typically", "usually", "generally",
    "likely", "possible", "based", "case", "risk", "symptoms", "years", "year",
    "multiple", "sclerosis",
}


def extract_keywords(text, max_keywords=8):
    words = re.findall(r"[A-Za-z0-9][A-Za-z0-9\-]*[A-Za-z0-9]", text)
    keywords = []
    for w in words:
        if w.lower() in STOPWORDS or len(w) < 3 or w.isdigit():
            continue
        if w.lower() not in [k.lower() for k in keywords]:
            keywords.append(w)
        if len(keywords) == max_keywords:
            break
    return keywords


def build_query(keywords):
    ms = '("multiple sclerosis"[MeSH Terms] OR "multiple sclerosis"[tiab])'
    terms = " OR ".join(f'"{k}"[tiab]' for k in keywords)
    filters = 'english[lang] AND hasabstract AND ("2010"[dp] : "3000"[dp])'
    return f"{ms} AND ({terms}) AND {filters}"


def fetch_abstracts(pmids):
    handle = Entrez.efetch(db="pubmed", id=",".join(pmids), retmode="xml")
    records = Entrez.read(handle)
    handle.close()
    papers = []
    for article in records.get("PubmedArticle", []):
        info = article["MedlineCitation"]["Article"]
        abstract = " ".join(str(p) for p in info.get("Abstract", {}).get("AbstractText", []))
        if abstract.strip():
            papers.append({"pmid": str(article["MedlineCitation"]["PMID"]),
                           "title": str(info.get("ArticleTitle", "")),
                           "abstract": abstract})
    return papers


def search_pubmed_v2(text, n_candidates=20, top_k=3, min_score=0.35, max_chars=1000):
    """Zwraca (evidence, citations_map, log). Gdy nic nie pasuje, evidence = ""."""
    log = {"keywords": "", "query": "", "n_candidates": 0, "pmids": "", "scores": "", "error": ""}
    try:
        keywords = extract_keywords(text)
        log["keywords"] = ", ".join(keywords)
        if not keywords:
            return "", {}, log
        log["query"] = build_query(keywords)

        handle = Entrez.esearch(db="pubmed", term=log["query"], retmax=n_candidates, sort="relevance")
        pmids = Entrez.read(handle)["IdList"]
        handle.close()
        log["n_candidates"] = len(pmids)
        if not pmids:
            return "", {}, log
        time.sleep(0.4)
        papers = fetch_abstracts(pmids)
        vectors = ranker.encode([text] + [p["title"] + ". " + p["abstract"] for p in papers],
                                normalize_embeddings=True)
        scores = vectors[1:] @ vectors[0]

        ranked = sorted(zip(scores, papers), key=lambda x: x[0], reverse=True)
        kept = [(s, p) for s, p in ranked if s >= min_score][:top_k]
        log["pmids"] = ";".join(p["pmid"] for _, p in kept)
        log["scores"] = ";".join(f"{s:.2f}" for s, _ in kept)

        evidence_blocks, citations_map = [], {}
        for i, (s, p) in enumerate(kept, start=1):
            citations_map[f"[SOURCE {i}]"] = f"the study '{p['title']}' (PMID: {p['pmid']})"
            evidence_blocks.append(f"--- [SOURCE {i}] ---\nTitle: {p['title']}\n{p['abstract'][:max_chars]}\n")
        return "\n".join(evidence_blocks), citations_map, log

    except Exception as e:
        log["error"] = f"{type(e).__name__}: {e}"
        return "", {}, log

In [ ]:
import pandas as pd

answers = pd.read_csv("answers_gemini.csv")

for i in [0, 8, 20]:
    text = answers["gemini_answer"].iloc[i]
    evidence, citations, log = search_pubmed_v2(text)

    print(f"Odpowiedź {i}")
    print(f"Tekst:          {text[:150]}...")
    print(f"Słowa kluczowe: {log['keywords']}")
    print(f"Kandydaci:      {log['n_candidates']} | wybrane: {log['pmids'] or 'brak'} | podobieństwo: {log['scores'] or '-'}")
    for tag, cite in citations.items():
        print(f"  {tag} {cite}")
    if log["error"]:
        print(f"Błąd: {log['error']}")
    print()